In [1]:
from datasets import load_dataset
import sqlparse

ds = load_dataset("xu3kev/BIRD-SQL-data-train")

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['db_id', 'question', 'evidence', 'SQL', 'schema'],
        num_rows: 9428
    })
})

In [3]:
ds["train"][0]

{'db_id': 'video_games',
 'question': 'List down at least five publishers of the games with number of sales less than 10000.',
 'evidence': 'publishers refers to publisher_name; number of sales less than 10000 refers to num_sales < 0.1;',
 'SQL': "SELECT T.publisher_name FROM ( SELECT DISTINCT T5.publisher_name FROM region AS T1 INNER JOIN region_sales AS T2 ON T1.id = T2.region_id INNER JOIN game_platform AS T3 ON T2.game_platform_id = T3.id INNER JOIN game_publisher AS T4 ON T3.game_publisher_id = T4.id INNER JOIN publisher AS T5 ON T4.publisher_id = T5.id WHERE T1.region_name = 'North America' AND T2.num_sales * 100000 < 10000 LIMIT 5 ) t",
 'schema': 'CREATE TABLE genre\n(\n    id         INTEGER not null\n            primary key,\n    genre_name TEXT default NULL\n);\nCREATE TABLE game\n(\n    id        INTEGER not null\n            primary key,\n    genre_id  INTEGER          default NULL,\n    game_name TEXT default NULL,\n    foreign key (genre_id) references genre(id)\n);\nCRE

In [4]:
join_count = 0
total_count = 0
non_join_queries = []

for row in ds["train"]:
    sql = row["SQL"]
    total_count += 1
    if "JOIN" in sql.upper():
        join_count += 1
    else:
        non_join_queries.append(sql)

print(f"Number of SQL statements containing JOIN: {join_count}")
print(f"Total number of SQL statements: {total_count}")
print(f"Percentage: {(join_count/total_count)*100:.2f}%")

print("\nSample of queries without JOINs:")
for i, query in enumerate(non_join_queries[:3]):  # Show first 3 non-join queries
    print(f"\nQuery {i+1}:")
    print(query)

Number of SQL statements containing JOIN: 7212
Total number of SQL statements: 9428
Percentage: 76.50%

Sample of queries without JOINs:

Query 1:
SELECT COUNT(name) FROM director WHERE director = 'Wolfgang Reitherman'

Query 2:
SELECT COUNT(bioguide) FROM `current-terms` WHERE class IS NULL

Query 3:
SELECT character_id FROM paragraphs WHERE PlainText = 'O my poor brother! and so perchance may he be.'


In [8]:
class SQLToWeaviateParser:
    def parse_sql(self, sql: str) -> dict:
        # Simple parsing for demonstration
        parts = sql.upper().split()
        query = {"collection_name": ""}
        
        # Get collection name
        try:
            from_index = parts.index("FROM")
            query["collection_name"] = parts[from_index + 1].lower()
        except:
            pass
            
        # Get where conditions
        try:
            where_index = parts.index("WHERE")
            property_name = parts[where_index + 1].lower()
            operator = parts[where_index + 2]
            value = parts[where_index + 3]
            
            try:
                value = float(value)
                query["integer_property_filter"] = {
                    "property_name": property_name,
                    "operator": operator,
                    "value": value
                }
            except ValueError:
                if value.lower() in ('true', 'false'):
                    query["boolean_property_filter"] = {
                        "property_name": property_name,
                        "operator": operator,
                        "value": value.lower() == 'true'
                    }
                else:
                    query["text_property_filter"] = {
                        "property_name": property_name,
                        "operator": operator,
                        "value": value
                    }
        except:
            pass
            
        return query

# Test
parser = SQLToWeaviateParser()
sql = "SELECT * FROM videos WHERE views > 1000"
print(parser.parse_sql(sql))

{'collection_name': 'videos', 'integer_property_filter': {'property_name': 'views', 'operator': '>', 'value': 1000.0}}


In [11]:
import json
import re

def sql_to_weaviate_schema(sql_schema):
    # Parse CREATE TABLE statements
    table_pattern = r"CREATE TABLE (\w+)\s*\((.*?)\);"
    tables = re.findall(table_pattern, sql_schema, re.DOTALL)
    
    collections = []
    for table_name, table_content in tables:
        # Parse columns
        column_pattern = r"(\w+)\s+(INTEGER|TEXT|REAL)(?:\s+not null)?(?:\s+primary key)?(?:\s+default NULL)?"
        columns = re.findall(column_pattern, table_content)
        
        properties = []
        for col_name, col_type in columns:
            if col_name == 'id':  # Skip id columns as Weaviate handles these automatically
                continue
                
            data_type = ["number"] if col_type in ["INTEGER", "REAL"] else ["string"]
            
            properties.append({
                "name": col_name,
                "data_type": data_type,
                "description": f"The {col_name.replace('_', ' ')} of the {table_name}."
            })
            
        if properties:  # Only add tables with properties
            collections.append({
                "name": table_name.capitalize(),
                "properties": properties,
                "envisioned_use_case_overview": f"This collection stores information about {table_name.replace('_', ' ')}s."
            })
    
    return {
        "weaviate_collections": collections
    }
# Process first schema and save
for row in ds["train"]:
    schema, SQL_query, nl_command = row["schema"], row["SQL"], row["question"]
    weaviate_schema = sql_to_weaviate_schema(schema)
    weaviate_query = parser.parse_sql(SQL_query)
    
    output = {
        "weaviate_collections": weaviate_schema["weaviate_collections"],
        "sql_query": SQL_query,
        "natural_language_query": nl_command
    }

    print(output)
    
    with open("bird-to-weaviate.json", "w") as f:
        json.dump(output, f, indent=2)
    break

{'weaviate_collections': [{'name': 'Genre', 'properties': [{'name': 'genre_name', 'data_type': ['string'], 'description': 'The genre name of the genre.'}], 'envisioned_use_case_overview': 'This collection stores information about genres.'}, {'name': 'Game', 'properties': [{'name': 'genre_id', 'data_type': ['number'], 'description': 'The genre id of the game.'}, {'name': 'game_name', 'data_type': ['string'], 'description': 'The game name of the game.'}], 'envisioned_use_case_overview': 'This collection stores information about games.'}, {'name': 'Platform', 'properties': [{'name': 'platform_name', 'data_type': ['string'], 'description': 'The platform name of the platform.'}], 'envisioned_use_case_overview': 'This collection stores information about platforms.'}, {'name': 'Publisher', 'properties': [{'name': 'publisher_name', 'data_type': ['string'], 'description': 'The publisher name of the publisher.'}], 'envisioned_use_case_overview': 'This collection stores information about publishe